In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer,from_unixtime




##############################
#######--SALES ORDERS--#######
##############################

def sales_orders(df):
    sales_orders = df.select("order_number",
                            "customer_id",
                            "customer_name",
                            "number_of_line_items",
                            from_unixtime(col("order_datetime")).alias("order_timestamps"))
    return sales_orders

##############################
######--ORDERD PRODUCTS--#####
##############################

def orderd_product(df):
        orderd_product_schema = ArrayType(
            StructType([
                StructField("curr", StringType()),
                StructField("id", StringType()),
                StructField("name", StringType()),
                StructField("price", StringType()),
                StructField("promotion_info", StringType()),
                StructField("qty", StringType()),
                StructField("unit", StringType())]
            ))

        df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), orderd_product_schema))\
                    .withColumn("ordered_products",explode_outer("ordered_products"))\
                    .select(
                        "customer_id",
                        "customer_name",
                        "order_number",
                        "ordered_products.id",
                        col("ordered_products.name").alias("product_name"),
                        "ordered_products.price",
                        "ordered_products.curr",
                        "ordered_products.qty",
                        "ordered_products.unit",
                    )
        return df_parsed


##############################
#########--PROMOTIONS--#######
##############################

from pyspark.sql.types import ArrayType

def promotions(df):
    promo_info_schema = ArrayType(StructType([
        StructField("promo_disc", StringType()),
        StructField("promo_id", StringType()),
        StructField("promo_item", StringType()),
        StructField("promo_qty", StringType())
    ]))
    df_promo_parsed = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))\
                        .withColumn("promo_info",explode_outer("promo_info")).filter(col("promo_info").isNotNull())\
                        .select(
                            "customer_id",
                            "customer_name",
                            "order_number",
                            "promo_info.*"              
                        )
    return df_promo_parsed


#########################
####--CLICKED ITEMS--####    
#########################

def clicked_items(df):
    
    clicked_items_schema = ArrayType(
        ArrayType(StringType())
            ) 


    clicked_item = df.withColumn("clicked_items",from_json(col("clicked_items"),clicked_items_schema))\
                .withColumn("clicked_items",explode_outer("clicked_items")).filter(col("clicked_items").isNotNull())\
                .select(
                    "customer_id",
                    "customer_name",
                    "order_number",
                    col("clicked_items")[0].alias("product_id"),
                    col("clicked_items")[1].alias("score")
                    )
    return clicked_item



 #####################
 ####--MAIN CODE--####    
 #####################                   
 
print(f"Reading source table")
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
 
print(f"Transforming data - Processing Sales Orders Table")
print(f"Writing data - Processing Sales Orders Table")
sales_orders(df).write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales_orders")

print(f"Transforming data - Processing Orderd Products Table")
print(f"Writing data - Processing Orderd Products Table")
orderd_product(df).write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.orderd_products")

print(f"Transforming data - Processing Promotions Table")
print(f"Writing data - Processing Promotions Table")
promotions(df).write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promo_info")
print(f"successfully wrote all tables")

print(f"Transforming data - Processing clicked items Table")
print(f"Writing data - Processing clicked items Table")
clicked_items(df).write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.clicked_items")

print(f"successfully wrote all tables")
